In [1]:
!pip -q install transformers datasets evaluate scikit-learn sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00


In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    set_seed
)

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
DATA_PATH = "/content/drive/MyDrive/dual_head_training_data.csv"
OUTPUT_DIR = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2"

MODEL_NAME = "xlm-roberta-large"

TEXT_COL = "model_input"
ROOT_LABEL_COL = "root_label"
SUFFIX_LABEL_COL = "suffix_label"

ROOT_TEXT_COL = "correct_root"
SUFFIX_TEXT_COL = "correct_suffix"

In [5]:

# Filtering
MIN_ROOT_FREQ = 3      # keep roots appearing at least this many times
MIN_SUFFIX_FREQ = 1    # usually keep all suffixes


In [6]:
TEST_SIZE = 0.10
VAL_SIZE = 0.10   # from full data, not from train
RANDOM_STATE = 42


In [7]:
MAX_LEN = 96
BATCH_SIZE = 32
NUM_EPOCHS = 6
LR = 2e-5
WEIGHT_DECAY = 0.01
ROOT_LOSS_WEIGHT = 3.0
SUFFIX_LOSS_WEIGHT = 1.0

set_seed(RANDOM_STATE)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
df = pd.read_csv(DATA_PATH)

print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())

# Drop missing important fields
df = df.dropna(subset=[TEXT_COL, ROOT_TEXT_COL, SUFFIX_TEXT_COL]).copy()

# Make sure text columns are strings
df[TEXT_COL] = df[TEXT_COL].astype(str)
df[ROOT_TEXT_COL] = df[ROOT_TEXT_COL].astype(str)
df[SUFFIX_TEXT_COL] = df[SUFFIX_TEXT_COL].astype(str)

print("Rows after dropna:", len(df))


Loaded rows: 90281
Columns: ['row_id', 'token_index', 'left_context', 'token', 'right_context', 'token_class', 'wrong_root', 'wrong_suffix', 'correct_root', 'correct_suffix', 'correct_token', 'generated_token', 'expected_token', 'model_input', 'root_label', 'suffix_label']
Rows after dropna: 17947


In [9]:
root_counts = df[ROOT_TEXT_COL].value_counts()
suffix_counts = df[SUFFIX_TEXT_COL].value_counts()

keep_roots = set(root_counts[root_counts >= MIN_ROOT_FREQ].index.tolist())
keep_suffixes = set(suffix_counts[suffix_counts >= MIN_SUFFIX_FREQ].index.tolist())

filtered_df = df[
    df[ROOT_TEXT_COL].isin(keep_roots) &
    df[SUFFIX_TEXT_COL].isin(keep_suffixes)
].copy()

print("\nAfter label-frequency filtering:")
print("Rows:", len(filtered_df))
print("Unique roots:", filtered_df[ROOT_TEXT_COL].nunique())
print("Unique suffixes:", filtered_df[SUFFIX_TEXT_COL].nunique())



After label-frequency filtering:
Rows: 17936
Unique roots: 796
Unique suffixes: 62


In [10]:
root_vocab = sorted(filtered_df[ROOT_TEXT_COL].unique().tolist())
suffix_vocab = sorted(filtered_df[SUFFIX_TEXT_COL].unique().tolist())

root2id = {label: i for i, label in enumerate(root_vocab)}
id2root = {i: label for label, i in root2id.items()}

suffix2id = {label: i for i, label in enumerate(suffix_vocab)}
id2suffix = {i: label for label, i in suffix2id.items()}

filtered_df["root_label_new"] = filtered_df[ROOT_TEXT_COL].map(root2id)
filtered_df["suffix_label_new"] = filtered_df[SUFFIX_TEXT_COL].map(suffix2id)

print("\nLabel space:")
print("Root labels:", len(root2id))
print("Suffix labels:", len(suffix2id))


Label space:
Root labels: 796
Suffix labels: 62


In [11]:
# Save label maps
with open(os.path.join(OUTPUT_DIR, "root2id.json"), "w", encoding="utf-8") as f:
    json.dump(root2id, f, ensure_ascii=False, indent=2)

with open(os.path.join(OUTPUT_DIR, "id2root.json"), "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in id2root.items()}, f, ensure_ascii=False, indent=2)

with open(os.path.join(OUTPUT_DIR, "suffix2id.json"), "w", encoding="utf-8") as f:
    json.dump(suffix2id, f, ensure_ascii=False, indent=2)

with open(os.path.join(OUTPUT_DIR, "id2suffix.json"), "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in id2suffix.items()}, f, ensure_ascii=False, indent=2)


In [12]:
train_val_df, test_df = train_test_split(
    filtered_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

# Then split train+val
val_ratio_from_trainval = VAL_SIZE / (1.0 - TEST_SIZE)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_ratio_from_trainval,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("\nSplit sizes:")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

train_df.to_csv(os.path.join(OUTPUT_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_split.csv"), index=False)



Split sizes:
Train: 14348
Val  : 1794
Test : 1794


In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
class DualHeadDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=96):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = row[TEXT_COL]
        root_label = int(row["root_label_new"])
        suffix_label = int(row["suffix_label_new"])

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "root_labels": torch.tensor(root_label, dtype=torch.long),
            "suffix_labels": torch.tensor(suffix_label, dtype=torch.long),
        }
        return item




In [15]:
train_dataset = DualHeadDataset(train_df, tokenizer, max_len=MAX_LEN)
val_dataset = DualHeadDataset(val_df, tokenizer, max_len=MAX_LEN)
test_dataset = DualHeadDataset(test_df, tokenizer, max_len=MAX_LEN)

In [16]:
class XLMRDualHeadModel(nn.Module):
    def __init__(self, model_name, num_root_labels, num_suffix_labels,
                 root_loss_weight=1.0, suffix_loss_weight=1.0):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.root_classifier = nn.Linear(hidden_size, num_root_labels)
        self.suffix_classifier = nn.Linear(hidden_size, num_suffix_labels)

        self.root_loss_fn = nn.CrossEntropyLoss()
        self.suffix_loss_fn = nn.CrossEntropyLoss()

        self.root_loss_weight = root_loss_weight
        self.suffix_loss_weight = suffix_loss_weight

    def forward(self, input_ids=None, attention_mask=None, root_labels=None, suffix_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]  # first token representation
        cls_repr = self.dropout(cls_repr)

        root_logits = self.root_classifier(cls_repr)
        suffix_logits = self.suffix_classifier(cls_repr)

        loss = None
        if root_labels is not None and suffix_labels is not None:
            root_loss = self.root_loss_fn(root_logits, root_labels)
            suffix_loss = self.suffix_loss_fn(suffix_logits, suffix_labels)
            loss = self.root_loss_weight * root_loss + self.suffix_loss_weight * suffix_loss

        return {
            "loss": loss,
            "root_logits": root_logits,
            "suffix_logits": suffix_logits
        }


In [17]:
model = XLMRDualHeadModel(
    model_name=MODEL_NAME,
    num_root_labels=len(root2id),
    num_suffix_labels=len(suffix2id),
    root_loss_weight=ROOT_LOSS_WEIGHT,
    suffix_loss_weight=SUFFIX_LOSS_WEIGHT
)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    root_logits, suffix_logits = predictions
    root_labels, suffix_labels = labels

    root_preds = np.argmax(root_logits, axis=1)
    suffix_preds = np.argmax(suffix_logits, axis=1)

    root_acc = accuracy_score(root_labels, root_preds)
    suffix_acc = accuracy_score(suffix_labels, suffix_preds)
    joint_acc = np.mean((root_preds == root_labels) & (suffix_preds == suffix_labels))
    suffix_macro_f1 = f1_score(suffix_labels, suffix_preds, average="macro")

    return {
        "root_accuracy": float(root_acc),
        "suffix_accuracy": float(suffix_acc),
        "joint_accuracy": float(joint_acc),
        "suffix_macro_f1": float(suffix_macro_f1),
    }

In [19]:
class DualHeadTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        root_labels = inputs.pop("root_labels")
        suffix_labels = inputs.pop("suffix_labels")

        outputs = model(**inputs, root_labels=root_labels, suffix_labels=suffix_labels)
        loss = outputs["loss"]

        if return_outputs:
            return loss, {
                "root_logits": outputs["root_logits"],
                "suffix_logits": outputs["suffix_logits"]
            }
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        has_labels = "root_labels" in inputs and "suffix_labels" in inputs

        inputs = self._prepare_inputs(inputs)

        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                root_labels=inputs["root_labels"] if has_labels else None,
                suffix_labels=inputs["suffix_labels"] if has_labels else None
            )

        loss = outputs["loss"].detach() if has_labels else None
        root_logits = outputs["root_logits"].detach()
        suffix_logits = outputs["suffix_logits"].detach()

        if prediction_loss_only:
            return (loss, None, None)

        # IMPORTANT: return torch tensors, not numpy arrays
        predictions = (root_logits, suffix_logits)

        if has_labels:
            labels = (inputs["root_labels"].detach(), inputs["suffix_labels"].detach())
        else:
            labels = None

        return (loss, predictions, labels)

In [20]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # REMOVE this line
    # overwrite_output_dir=True,

    eval_strategy="epoch",   # FIX name
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="joint_accuracy",
    greater_is_better=True,

    fp16=True,
    report_to="none"
)

In [21]:
trainer = DualHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [22]:
train_result = trainer.train()

print("\nTraining finished.")
print(train_result)

Epoch,Training Loss,Validation Loss,Root Accuracy,Suffix Accuracy,Joint Accuracy,Suffix Macro F1
1,12.266556,7.023330,0.785396,0.862319,0.688406,0.193716
2,4.971909,2.566580,0.948718,0.921962,0.885173,0.464417
3,2.382679,1.438402,0.964883,0.951505,0.929766,0.692132
4,1.376819,1.002977,0.968785,0.963768,0.944259,0.826351
5,0.848064,0.832996,0.971014,0.969900,0.951505,0.837790
6,0.662898,0.780179,0.972687,0.971572,0.954849,0.843235



Training finished.
TrainOutput(global_step=2694, training_loss=4.2879207316556505, metrics={'train_runtime': 475.0134, 'train_samples_per_second': 181.233, 'train_steps_per_second': 5.671, 'total_flos': 0.0, 'train_loss': 4.2879207316556505, 'epoch': 6.0})


In [23]:
trainer.save_model(os.path.join(OUTPUT_DIR, "best_model"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))


('/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/best_model/tokenizer_config.json',
 '/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/best_model/tokenizer.json')

In [24]:
val_metrics = trainer.evaluate(eval_dataset=val_dataset)
print("\nValidation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v}")


Validation metrics:
eval_loss: 0.7801791429519653
eval_root_accuracy: 0.9726867335562988
eval_suffix_accuracy: 0.9715719063545151
eval_joint_accuracy: 0.9548494983277592
eval_suffix_macro_f1: 0.8432352727473363
eval_runtime: 2.242
eval_samples_per_second: 800.172
eval_steps_per_second: 25.424
epoch: 6.0


In [ ]:

# Save metrics
with open(os.path.join(OUTPUT_DIR, "test_metrics.json"), "w") as f:
    json.dump(test_metrics, f, indent=2)


NameError: name 'test_metrics' is not defined

In [25]:
pred_output = trainer.predict(test_dataset)

(root_logits, suffix_logits) = pred_output.predictions
(root_true, suffix_true) = pred_output.label_ids

root_pred_ids = np.argmax(root_logits, axis=1)
suffix_pred_ids = np.argmax(suffix_logits, axis=1)

test_results_df = test_df.reset_index(drop=True).copy()
test_results_df["pred_root_id"] = root_pred_ids
test_results_df["pred_suffix_id"] = suffix_pred_ids

test_results_df["pred_root"] = test_results_df["pred_root_id"].map(id2root)
test_results_df["pred_suffix"] = test_results_df["pred_suffix_id"].map(id2suffix)

test_results_df["root_correct"] = (
    test_results_df["pred_root"] == test_results_df[ROOT_TEXT_COL]
)
test_results_df["suffix_correct"] = (
    test_results_df["pred_suffix"] == test_results_df[SUFFIX_TEXT_COL]
)
test_results_df["joint_correct"] = (
    test_results_df["root_correct"] & test_results_df["suffix_correct"]
)

def rebuild_token(row):
    if row["token_class"] == "EN":
        return row["pred_root"]
    return f"{row['pred_root']}-{row['pred_suffix']}"

test_results_df["pred_corrected_token"] = test_results_df.apply(rebuild_token, axis=1)

test_results_df.to_csv(os.path.join(OUTPUT_DIR, "test_predictions_detailed.csv"), index=False)

print("\nSaved detailed predictions to:")
print(os.path.join(OUTPUT_DIR, "test_predictions_detailed.csv"))


Saved detailed predictions to:
/content/drive/MyDrive/xlmr_large_dual_head_model_try_2/test_predictions_detailed.csv


In [26]:
print("\nRoot Accuracy:", accuracy_score(root_true, root_pred_ids))
print("Suffix Accuracy:", accuracy_score(suffix_true, suffix_pred_ids))
print("Joint Accuracy:", np.mean((root_pred_ids == root_true) & (suffix_pred_ids == suffix_true)))
print("Suffix Macro F1:", f1_score(suffix_true, suffix_pred_ids, average="macro"))

# Optional: suffix report
suffix_report = classification_report(
    suffix_true,
    suffix_pred_ids,
    target_names=[id2suffix[i] for i in range(len(id2suffix))],
    zero_division=0
)

with open(os.path.join(OUTPUT_DIR, "suffix_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(suffix_report)

print("\nSaved suffix classification report.")
print("Done.")


Root Accuracy: 0.9682274247491639
Suffix Accuracy: 0.9749163879598662
Joint Accuracy: 0.9542920847268673
Suffix Macro F1: 0.8978699598334506


ValueError: Number of classes, 54, does not match size of target_names, 62. Try specifying the labels parameter